# 符号定时同步 (Symbol Timing Synchronization)

## 学习目标

1. 理解符号定时偏移（STO）的概念及其对通信系统的影响
2. 掌握 Early-Late 门同步算法的原理与实现
3. 理解 Gardner 算法的核心思想（基于眼图）
4. 熟悉插值滤波器（线性、立方）的设计与应用
5. 掌握定时同步环路的基本结构

## 内容概览

| 主题 | 核心概念 | 应用场景 |
|:---|:---|:---|
| STO 基础 | 定时偏移、采样相位 | 接收机前端 |
| Early-Late 门 | 对称采样、误差检测 | 符号同步 |
| Gardner 算法 | 眼图分析、内插调整 | 高速通信 |
| 插值滤波器 | 分数延迟、采样率转换 | 数字下变频 |
| 定时环路 | 鉴相器、环路滤波器、NCO | 闭环同步 |

---

## 1. 符号定时偏移（STO）基础

### 1.1 什么是符号定时偏移？

符号定时偏移（Symbol Timing Offset, STO）是指接收机采样时刻与实际符号最佳采样点之间的时间差。在理想情况下，接收机应在每个符号的峰值时刻进行采样，但实际系统中：

- 发射机与接收机使用独立的时钟源，存在固有偏差
- 信号在传输过程中经历多径衰落，导致有效采样点偏移
- 采样率转换、滤波等处理会引入群延迟波动

### 1.2 STO 的数学表示

设符号周期为 $T_s$，采样周期为 $T_{sample}$，则第 $k$ 个符号的理想采样时刻为：

$$
t_k^{ideal} = k T_s + \tau_0
$$

其中 $\tau_0$ 是固定的定时偏移。当存在动态偏移时：

$$
t_k^{ideal} = k T_s + \tau_0 + \Delta \tau_k
$$

### 1.3 STO 对信号的影响

STO Effect

左侧：理想采样点 | 右侧：存在 STO 时的采样（符号间干扰 ISI）

当采样点偏离最佳位置时，会导致：

1. **符号间干扰（ISI）**: 采样点落在相邻符号的拖尾上
2. **信噪比恶化**: 采样信号包含噪声与干扰的叠加
3. **判决错误率上升**: 采样值偏离实际符号幅度

### 1.4 STO 估计方法分类

| 方法类型 | 代表算法 | 复杂度 | 精度 |
|:---|:---|:---:|:---:|
| 数据辅助 | 前导码相关检测 | 中 | 高 |
| 非数据辅助 | FFT 谱分析 | 高 | 中 |
| 判决导向 | Early-Late、Gardner | 低 | 中 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.interpolate import interp1d

# 设置中文字体支持
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 符号定时同步演示参数
Fs = 100  # 采样率（samples/symbol）
Ts = 1.0  # 符号周期
num_symbols = 50  # 符号数量
SNR = 20  # 信噪比(dB)

np.random.seed(42)
print(f"采样率: {Fs} samples/symbol")
print(f"符号周期: {Ts} s")
print(f"符号数量: {num_symbols}")
print(f"信噪比: {SNR} dB")

---

## 2. Early-Late 门同步算法

### 2.1 算法原理

Early-Late 门算法是一种基于能量检测的符号同步方法。其核心思想是：

- **Early 采样点**：在符号周期前端 $\tau - \delta$ 处采样
- **Late 采样点**：在符号周期后端 $\tau + \delta$ 处采样
- 当两者能量相等时，当前采样点 $\tau$ 即为符号中心

### 2.2 误差函数

$$e_k = |y_{k,E}|^2 - |y_{k,L}|^2$$

其中：

- $y_{k,E}$: Early 采样值（第 k 个符号前 $\delta$ 处）
- $y_{k,L}$: Late 采样值（第 k 个符号后 $\delta$ 处）

当 $e_k > 0$ 时，说明 Early 能量大于 Late，应增大采样相位；
当 $e_k < 0$ 时，应减小采样相位。

In [ ]:
def early_late_sync(signal_in, sps, early_late_spacing=0.5, loop_bw=0.01):
    """
    Early-Late 门同步算法
    
    参数:
    -----------
    signal_in : array-like
        输入信号序列
    sps : int
        每符号采样数 (samples per symbol)
    early_late_spacing : float
        Early/Late 采样点间距（符号周期倍数）
    loop_bw : float
        环路带宽（控制收敛速度）
    
    返回:
    -------
    synced_signal : array
        同步后的信号
    timing_error : array
        定时误差序列
    """
    n = len(signal_in)
    
    # 初始化采样相位
    mu = 0.0  # 分数间隔采样位置 [0, 1)
    
    synced_samples = []
    timing_error = []
    
    # Early/Late 采样间隔（samples）
    spacing = early_late_spacing * sps
    
    for i in range(sps, n - int(spacing) - 1):
        # 计算当前整数采样位置
        sample_idx = int(i + mu)
        
        # Early 采样点 (提前 spacing/2)
        early_idx = int(sample_idx - spacing / 2)
        # Late 采样点 (滞后 spacing/2)
        late_idx = int(sample_idx + spacing / 2)
        
        # 确保索引有效
        if early_idx < 0 or late_idx >= n:
            continue
        
        # Early 和 Late 能量计算
        early_val = signal_in[early_idx]
        late_val = signal_in[late_idx]
        
        early_energy = np.abs(early_val) ** 2
        late_energy = np.abs(late_val) ** 2
        
        # 定时误差
        error = early_energy - late_energy
        timing_error.append(error)
        
        # 更新采样相位（鉴相器输出）
        mu = mu - loop_bw * error
        
        # 限制 mu 在合理范围 [-1, 1)
        mu = np.clip(mu, -1, 1)
        
        # 保存同步采样值
        if 0 <= sample_idx < n:
            synced_samples.append(signal_in[sample_idx])
    
    return np.array(synced_samples), np.array(timing_error)

print("Early-Late 同步算法已定义")

In [ ]:
# 生成测试信号：QPSK 调制
def generate_qpsk_signal(num_sym, sps, carrier_phase=0, noise_var=0.1):
    """
    生成 QPSK 测试信号
    
    参数:
    -----------
    num_sym : int
        符号数量
    sps : int
        每符号采样数
    carrier_phase : float
        载波相位偏移（弧度）
    noise_var : float
        噪声方差
    
    返回:
    -------
    signal_out : array
        调制后的基带信号
    symbols : array
        原始符号序列
    """
    # QPSK 符号映射 (Gray编码)
    # 00 -> 1+1j, 01 -> -1+1j, 10 -> 1-1j, 11 -> -1-1j
    symbol_map = {
        (0, 0): 1 + 1j,
        (0, 1): -1 + 1j,
        (1, 0): 1 - 1j,
        (1, 1): -1 - 1j
    }
    
    # 生成随机符号
    bits = np.random.randint(0, 2, (num_sym, 2))
    symbols = np.array([symbol_map[tuple(b)] for b in bits])
    
    # 脉冲成形 (根升余弦滤波器)
    alpha = 0.35  # 滚降系数
    N = 10  # 滤波器长度（符号周期数）
    t = np.arange(-N * sps // 2, N * sps // 2 + 1) / sps
    
    # 根升余弦滤波器系数
    b = np.zeros(len(t))
    for i, tt in enumerate(t):
        if tt == 0:
            b[i] = (1 - alpha + 4 * alpha / np.pi) * np.sqrt(1 / sps)
        elif np.abs(np.abs(4 * alpha * tt / Ts) - 1) < 1e-10:
            b[i] = (alpha / np.sqrt(2)) * ((1 + 2 / np.pi) * np.sin(np.pi / (4 * alpha)) +
                                          (1 - 2 / np.pi) * np.cos(np.pi / (4 * alpha))) * np.sqrt(1 / sps)
        else:
            b[i] = (np.sin(np.pi * tt / Ts * (1 - alpha)) +
                    4 * alpha * tt / Ts * np.cos(np.pi * tt / Ts * (1 + alpha))) /
                   (np.pi * tt / Ts * (1 - (4 * alpha * tt / Ts) ** 2)) * np.sqrt(1 / sps)
    
    # 插值上采样
    upsampled = np.zeros(num_sym * sps)
    upsampled[::sps] = symbols
    
    # 卷积成形
    signal_filtered = np.convolve(upsampled, b, mode='same')
    
    # 添加噪声
    noise = np.sqrt(noise_var / 2) * (np.random.randn(len(signal_filtered)) +
                                        1j * np.random.randn(len(signal_filtered)))
    signal_out = signal_filtered + noise
    
    return signal_out, symbols

# 生成 QPSK 信号
sps = 16  # 每符号16个采样点
num_sym = 100  # 100个符号
qpsk_signal, tx_symbols = generate_qpsk_signal(num_sym, sps, noise_var=0.01)

print(f"生成信号长度: {len(qpsk_signal)} 采样点")
print(f"符号数量: {len(tx_symbols)}")
print(f"每符号采样数: {sps}")

In [ ]:
# 模拟 STO：人为引入定时偏移
sto_offset = 0.3  # 定时偏移（符号周期倍数）
offset_samples = int(sto_offset * sps)

# 将信号偏移半个符号周期
signal_with_sto = np.roll(qpsk_signal, offset_samples)

# 应用 Early-Late 同步
synced_signal, timing_err = early_late_sync(signal_with_sto, sps,
                                             early_late_spacing=0.5,
                                             loop_bw=0.02)

# 绘图：定时误差收敛过程
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 子图1：原始信号（受STO影响）
ax1 = axes[0, 0]
ax1.plot(np.real(qpsk_signal[:200]), 'b-', alpha=0.7, label='原始信号')
ax1.plot(np.real(signal_with_sto[:200]), 'r-', alpha=0.5, label='STO后信号')
ax1.set_xlabel('采样点')
ax1.set_ylabel('幅度')
ax1.set_title('原始信号 vs 定时偏移信号')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 子图2：定时误差收敛曲线
ax2 = axes[0, 1]
ax2.plot(timing_err[:500], 'g-', linewidth=0.8)
ax2.axhline(y=0, color='k', linestyle='--', alpha=0.5)
ax2.set_xlabel('迭代次数')
ax2.set_ylabel('定时误差')
ax2.set_title('Early-Late 定时误差收敛')
ax2.grid(True, alpha=0.3)

# 子图3：同步后信号星座图
ax3 = axes[1, 0]
synced_complex = synced_signal[::sps][:len(tx_symbols)]
ax3.plot(np.real(synced_complex), np.imag(synced_complex), 'bo', markersize=4, alpha=0.6)
ax3.plot([1, -1, -1, 1], [1, 1, -1, -1], 'r+', markersize=10)  # 理想点
ax3.set_xlabel('I（同相）')
ax3.set_ylabel('Q（正交）')
ax3.set_title('同步后 QPSK 星座图')
ax3.grid(True, alpha=0.3)
ax3.set_xlim(-1.5, 1.5)
ax3.set_ylim(-1.5, 1.5)

# 子图4：眼图（同步后）
ax4 = axes[1, 1]
for i in range(10):
    start = i * sps
    end = start + 2 * sps
    if end < len(synced_signal):
        ax4.plot(np.real(synced_signal[start:end]), 'b-', alpha=0.4)
ax4.set_xlabel('采样点（符号周期）')
ax4.set_ylabel('幅度')
ax4.set_title('同步后眼图')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('early_late_sync_demo.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nEarly-Late 同步演示完成")

### 2.3 Early-Late 门算法特点

| 优点 | 缺点 |
|:---|:---|
| 实现简单，复杂度低 | 对噪声敏感 |
| 不需要先验信息 | 需要足够的数据符号数 |
| 收敛速度较快 | 精度有限（受spacing参数影响）|
| 适用于各种调制方式 | 存在相位模糊问题 |

---

## 3. Gardner 算法

### 3.1 算法背景

Gardner 算法由 Donald D. Gardner 于 1986 年提出，是一种基于眼图分析的判决导向定时同步算法。与 Early-Late 不同，Gardner 算法利用两个采样点（每位符号一个）来检测定时误差。

### 3.2 核心原理

Gardner 算法的误差函数为：

$$
e_k = \text{sgn}(y_{I,k}) \cdot (y_{Q,k} - y_{Q,k-1}) + \text{sgn}(y_{Q,k}) \cdot (y_{I,k} - y_{I,k-1})
$$

其中：

- $y_{I,k}$: 第 k 个符号的 I 分量采样值
- $y_{Q,k}$: 第 k 个符号的 Q 分量采样值
- $\text{sgn}(\cdot)$: 符号函数

### 3.3 简化形式（用于 QPSK）

对于 QPSK 调制，Gardner 误差简化为：

$$
e_k = \text{sgn}(y_k) \cdot (y_{k-1} - y_{k+1})
$$

当采样时刻准确时，$y_{k-1} = y_{k+1}$，误差为零。
当采样过早时，$y_{k-1} > y_{k+1}$；当采样过晚时，$y_{k-1} < y_{k+1}$。

In [ ]:
def gardner_sync(signal_in, sps, loop_bw=0.01, max_ctr=200):
    """
    Gardner 定时同步算法
    
    参数:
    -----------
    signal_in : array-like
        输入复基带信号
    sps : int
        每符号采样数
    loop_bw : float
        环路带宽
    max_ctr : int
        最大迭代次数
    
    返回:
    -------
    synced_signal : array
        同步后信号
    timing_error : array
        定时误差序列
    mu_history : array
        采样相位变化历史
    """
    n = len(signal_in)
    
    # 采样相位初始化
    mu = 0.5  # 从符号中心开始
    
    synced_samples = []
    timing_error = []
    mu_history = [mu]
    
    # 遍历所有可能的符号位置
    for k in range(1, n // sps - 1):
        # 计算当前符号的采样位置
        sample_idx = k * sps + int(mu * sps)
        
        if sample_idx >= n - 1:
            break
        
        # 获取当前符号及前后符号的采样值
        y_k = signal_in[sample_idx]
        y_prev = signal_in[sample_idx - sps]
        y_next = signal_in[sample_idx + sps]
        
        # Gardner 误差计算（简化形式）
        # e_k = sgn(y_k) * (y_{k-1} - y_{k+1})
        if np.abs(y_k) > 1e-6:  # 避免零除
            error = np.sign(np.real(y_k)) * (np.real(y_prev) - np.real(y_next)) + \
                    np.sign(np.imag(y_k)) * (np.imag(y_prev) - np.imag(y_next))
        else:
            error = 0
        
        timing_error.append(error)
        
        # 更新采样相位
        mu = mu - loop_bw * error
        
        # 相位归一化
        mu = mu % 1.0
        mu_history.append(mu)
        
        # 保存同步采样值
        if sample_idx < n:
            synced_samples.append(y_k)
    
    return np.array(synced_samples), np.array(timing_error), np.array(mu_history)

print("Gardner 同步算法已定义")

In [ ]:
# 应用 Gardner 算法
sto_offset_g = 0.4
offset_samples_g = int(sto_offset_g * sps)
signal_with_sto_g = np.roll(qpsk_signal, offset_samples_g)

# 运行 Gardner 同步
synced_g, err_g, mu_hist = gardner_sync(signal_with_sto_g, sps, loop_bw=0.015)

# 绘图对比
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 子图1：原始信号与STO信号
ax1 = axes[0, 0]
ax1.plot(np.real(signal_with_sto_g[:200]), 'r-', alpha=0.6, label='STO信号')
ax1.plot(np.real(qpsk_signal[:200]), 'b--', alpha=0.5, label='原始信号')
ax1.set_xlabel('采样点')
ax1.set_ylabel('幅度')
ax1.set_title('Gardner算法 - STO信号 vs 原始信号')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 子图2：Gardner误差收敛
ax2 = axes[0, 1]
ax2.plot(err_g[:500], 'g-', linewidth=0.8, label='Gardner误差')
ax2.axhline(y=0, color='k', linestyle='--', alpha=0.5)
ax2.set_xlabel('符号索引')
ax2.set_ylabel('误差值')
ax2.set_title('Gardner定时误差收敛')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 子图3：采样相位演化
ax3 = axes[1, 0]
ax3.plot(mu_hist[:500], 'm-', linewidth=1.0)
ax3.axhline(y=0.5, color='k', linestyle='--', alpha=0.5, label='理想采样点')
ax3.set_xlabel('迭代次数')
ax3.set_ylabel('采样相位 mu')
ax3.set_title('采样相位收敛过程')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 子图4：同步后星座图
ax4 = axes[1, 1]
synced_complex_g = synced_g[::sps][:len(tx_symbols)]
ax4.plot(np.real(synced_complex_g), np.imag(synced_complex_g), 'bo', markersize=4, alpha=0.6)
ax4.plot([1, -1, -1, 1], [1, 1, -1, -1], 'r+', markersize=10)
ax4.set_xlabel('I（同相）')
ax4.set_ylabel('Q（正交）')
ax4.set_title('Gardner同步后星座图')
ax4.grid(True, alpha=0.3)
ax4.set_xlim(-1.5, 1.5)
ax4.set_ylim(-1.5, 1.5)

plt.tight_layout()
plt.savefig('gardner_sync_demo.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nGardner 同步演示完成")

### 3.4 Gardner vs Early-Late 对比

| 特性 | Gardner | Early-Late |
|:---|:---|:---|
| 采样点数/符号 | 1（眼图中心） | 2（Early/Late） |
| 误差检测 | 符号翻转检测 | 能量差检测 |
| 复杂度 | 中等 | 低 |
| 噪声敏感性 | 较低 | 较高 |
| 适用场景 | QPSK/高阶调制 | 低阶调制 |
| 载波相位影响 | 不敏感 | 不敏感 |

---

## 4. 插值滤波器

### 4.1 为什么需要插值滤波器？

在数字通信接收机中，模数转换器（ADC）以固定采样率 $F_s$ 进行采样，而符号周期 $T_s$ 通常不是 $F_s$ 的整数倍。此外，定时同步算法产生的采样相位误差 $\mu$ 是分数间隔的。

插值滤波器的作用是：根据新的采样相位 $\mu_k$，从现有采样序列中计算得到最佳采样值。

### 4.2 线性插值

线性插值是最简单的插值方法。给定两个相邻采样点 $y[n]$ 和 $y[n+1]$，以及分数间隔 $\mu \in [0, 1)$，插值结果为：

$$
y_{int}[k] = (1 - \mu) \cdot y[n] + \mu \cdot y[n+1]
$$

线性插值的频率响应为 $\text{sinc}$ 函数，存在一定的带内失真，但对于低阶调制（QPSK等）已足够。

In [ ]:
def linear_interpolator(signal_in, mu_array):
    """
    线性插值滤波器
    
    参数:
    -----------
    signal_in : array
        输入信号序列
    mu_array : array
        分数间隔序列，每个值在 [0, 1)
    
    返回:
    -------
    y_interp : array
        插值后的信号序列
    """
    n = len(signal_in)
    y_interp = []
    
    for mu in mu_array:
        # 确定整数部分索引
        n_idx = int(mu)
        # 确定下一个采样点索引
        n_plus_1 = min(n_idx + 1, n - 1)
        
        # 线性插值
        y_val = (1 - mu + n_idx) * signal_in[n_idx] + (mu - n_idx) * signal_in[n_plus_1]
        y_interp.append(y_val)
    
    return np.array(y_interp)


def linear_interp_vectorized(signal_in, mu_array):
    """
    线性插值（向量化版本，更高效）
    
    参数:
    -----------
    signal_in : array
        输入信号
    mu_array : array
        分数间隔数组
    
    返回:
    -------
    y_interp : array
        插值结果
    """
    # 将 mu 转换为整数索引和分数部分
    n_idx = np.floor(mu_array).astype(int)
    frac = mu_array - n_idx
    
    # 边界处理
    n_idx = np.clip(n_idx, 0, len(signal_in) - 2)
    
    # 向量化线性插值
    y_interp = (1 - frac) * signal_in[n_idx] + frac * signal_in[n_idx + 1]
    
    return y_interp

print("线性插值滤波器已定义")

### 4.3 立方插值

立方插值利用4个相邻采样点拟合三次多项式，精度更高：

$$
y_{int}[k] = \sum_{i=-1}^{2} c_i \cdot y[n+i]
$$

其中多项式系数 $c_i$ 由分数间隔 $\mu$ 决定。

立方插值的频率响应更接近理想低通，在高阶调制场景中表现更好。

In [ ]:
def cubic_interpolator(signal_in, mu_array):
    """
    立方插值滤波器（使用 Catmull-Rom 样条）
    
    参数:
    -----------
    signal_in : array
        输入信号序列
    mu_array : array
        分数间隔数组 [0, 1)
    
    返回:
    -------
    y_interp : array
        插值结果
    """
    n = len(signal_in)
    y_interp = []
    
    for mu in mu_array:
        # 确定采样点索引（mu=0 对应 y[0]）
        n_idx = int(mu)
        # 分数部分
        frac = mu - n_idx
        
        # 确保有足够的采样点
        if n_idx >= n - 1:
            y_interp.append(signal_in[-1])
            continue
        
        # 获取4个相邻点（边界处理）
        y_m1 = signal_in[max(0, n_idx - 1)]
        y_0 = signal_in[n_idx]
        y_1 = signal_in[min(n - 1, n_idx + 1)]
        y_2 = signal_in[min(n - 1, n_idx + 2)]
        
        # Catmull-Rom 插值公式
        # P(t) = 0.5 * ((2*y_0) + (-y_m1 + y_1)*t + (2*y_m1 - 5*y_0 + 4*y_1 - y_2)*t^2 + (-y_m1 + 3*y_0 - 3*y_1 + y_2)*t^3)
        t = frac
        y_val = 0.5 * (
            (2 * y_0) +
            (-y_m1 + y_1) * t +
            (2 * y_m1 - 5 * y_0 + 4 * y_1 - y_2) * t ** 2 +
            (-y_m1 + 3 * y_0 - 3 * y_1 + y_2) * t ** 3
        )
        y_interp.append(y_val)
    
    return np.array(y_interp)


# 测试插值效果
print("立方插值滤波器已定义")

In [ ]:
# 对比线性插值与立方插值
t_original = np.arange(0, 10, 0.5)  # 原始采样点
y_original = np.sin(2 * np.pi * t_original / 10) + 0.1 * np.random.randn(len(t_original))

# 目标采样位置（非整数倍）
t_interp = np.arange(0, 10, 0.1)  # 更高分辨率

# 使用不同插值方法
mu_test = t_interp / 0.5  # 转换为mu格式
y_linear = linear_interp_vectorized(y_original, mu_test)
y_cubic = cubic_interpolator(y_original, mu_test)

# 绘图对比
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左图：插值效果对比
ax1 = axes[0]
ax1.plot(t_original, y_original, 'ko', markersize=8, label='原始采样点', zorder=5)
ax1.plot(t_interp, y_linear, 'b-', alpha=0.7, label='线性插值', linewidth=2)
ax1.plot(t_interp, y_cubic, 'r--', alpha=0.8, label='立方插值', linewidth=2)
ax1.set_xlabel('时间 (normalized)')
ax1.set_ylabel('幅度')
ax1.set_title('线性插值 vs 立方插值')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 右图：误差对比
ax2 = axes[1]
y_true = np.sin(2 * np.pi * t_interp / 10)
error_linear = y_linear - y_true
error_cubic = y_cubic - y_true
ax2.plot(t_interp, error_linear, 'b-', alpha=0.7, label='线性插值误差')
ax2.plot(t_interp, error_cubic, 'r--', alpha=0.8, label='立方插值误差')
ax2.axhline(y=0, color='k', linestyle='--', alpha=0.5)
ax2.set_xlabel('时间 (normalized)')
ax2.set_ylabel('插值误差')
ax2.set_title('插值误差对比')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('interpolation_compare.png', dpi=150, bbox_inches='tight')
plt.show()

# 计算均方根误差
rmse_linear = np.sqrt(np.mean(error_linear ** 2))
rmse_cubic = np.sqrt(np.mean(error_cubic ** 2))
print(f"线性插值 RMSE: {rmse_linear:.4f}")
print(f"立方插值 RMSE: {rmse_cubic:.4f}")

### 4.4 插值滤波器特性对比

| 特性 | 线性插值 | 立方插值 |
|:---|:---|:---|
| 计算复杂度 | 低 O(n) | 中 O(n) |
| 延迟 | 0.5 符号 | 1.5 符号 |
| 带内平坦度 | 较差 | 较好 |
| 旁瓣抑制 | 较差 | 中等 |
| 适用场景 | 低成本实现 | 高精度需求 |

---

## 5. 定时同步环路

### 5.1 环路结构

定时同步环路（Timing Recovery Loop）是闭环反馈系统，包含以下模块：

```
    输入信号 ──▶ 插值滤波器 ──▶ 符号采样 ──┬──▶ 鉴相器 ──▶ 环路滤波器 ──▶ NCO ──▶ 采样相位控制
                                       │                              ▲
                                       │                              │
                                       └──────────────────────────────┘
```

### 5.2 核心模块

| 模块 | 功能 | 常见实现 |
|:---|:---|:---|
| **鉴相器** | 将采样相位误差转换为误差信号 | Gardner, Early-Late, Mueller |
| **环路滤波器** | 平滑误差信号，抑制噪声 | 一阶IIR、二阶PI |
| **NCO** | 数值控制器振荡器，产生采样相位 | 积分器 + 相位累加器 |

In [ ]:
class TimingRecoveryLoop:
    """
    定时同步环路实现
    
    包含：鉴相器(Gardner) + 环路滤波器 + NCO
    """
    
    def __init__(self, sps, loop_bw=0.01, damping=0.707):
        """
        初始化定时同步环路
        
        参数:
        -------
        sps : int
            每符号采样数
        loop_bw : float
            环路带宽（归一化）
        damping : float
            阻尼系数
        """
        self.sps = sps
        self.loop_bw = loop_bw
        self.damping = damping
        
        # NCO状态
        self.phase = 0.5  # 当前采样相位
        self.freq = 0.0   # NCO频率控制字
        
        # 环路滤波器状态（一阶IIR）
        self.integrator = 0.0
        
        # 环路增益
        self.gain = loop_bw
        
        # 误差历史
        self.error_history = []
        self.phase_history = []
    
    def phase_detector(self, y_prev, y_curr, y_next):
        """
        Gardner 鉴相器
        
        参数:
        -------
        y_prev, y_curr, y_next : complex
            前一、当前、后一符号采样值
        
        返回:
        -------
        error : float
            定时误差
        """
        if np.abs(y_curr) < 1e-6:
            return 0
        
        # Gardner误差公式
        error = np.sign(np.real(y_curr)) * (np.real(y_prev) - np.real(y_next)) + \
                np.sign(np.imag(y_curr)) * (np.imag(y_prev) - np.imag(y_next))
        
        return error
    
    def loop_filter(self, error):
        """
        环路滤波器（比例+积分）
        
        参数:
        -------
        error : float
            鉴相器输出
        
        返回:
        -------
        u : float
            滤波器输出
        """
        # 比例项
        prop = error * self.gain
        
        # 积分项
        self.integrator += error * self.gain * 0.1
        
        # 总输出
        u = prop + self.integrator
        
        return u
    
    def nco_update(self, u):
        """
        NCO 更新（积分器）
        
        参数:
        -------
        u : float
            环路滤波器输出
        """
        # 更新NCO相位
        self.phase += u
        
        # 限制相位在 [0, 1)
        self.phase = self.phase % 1.0
        
        return self.phase
    
    def process(self, signal_in):
        """
        处理输入信号
        
        参数:
        -------
        signal_in : array
            输入信号
        
        返回:
        -------
        synced_samples : array
            同步后的采样序列
        """
        n = len(signal_in)
        synced_samples = []
        
        # 跳过第一个和最后一个符号
        for k in range(1, n // self.sps - 1):
            # 计算当前符号的采样索引
            base_idx = k * self.sps
            sample_idx = base_idx + int(self.phase * self.sps)
            
            if sample_idx >= n - 1:
                break
            
            # 获取三个采样点
            y_prev = signal_in[sample_idx - self.sps]
            y_curr = signal_in[sample_idx]
            y_next = signal_in[sample_idx + self.sps]
            
            # 鉴相
            error = self.phase_detector(y_prev, y_curr, y_next)
            
            # 环路滤波
            u = self.loop_filter(error)
            
            # NCO更新
            self.phase = self.nco_update(u)
            
            # 保存误差和相位历史
            self.error_history.append(error)
            self.phase_history.append(self.phase)
            
            # 保存同步采样值
            synced_samples.append(y_curr)
        
        return np.array(synced_samples)

# 创建定时同步环路实例
loop = TimingRecoveryLoop(sps=16, loop_bw=0.02)
print("定时同步环路已创建")

In [ ]:
# 使用定时同步环路处理信号
sto_offset_loop = 0.35
offset_samples_loop = int(sto_offset_loop * sps)
signal_sto_loop = np.roll(qpsk_signal, offset_samples_loop)

# 处理信号
synced_loop = loop.process(signal_sto_loop)

# 绘图
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 子图1：误差收敛
ax1 = axes[0, 0]
ax1.plot(loop.error_history[:500], 'g-', linewidth=0.8)
ax1.axhline(y=0, color='k', linestyle='--', alpha=0.5)
ax1.set_xlabel('符号索引')
ax1.set_ylabel('鉴相器误差')
ax1.set_title('Gardner鉴相器误差收敛')
ax1.grid(True, alpha=0.3)

# 子图2：相位收敛
ax2 = axes[0, 1]
ax2.plot(loop.phase_history[:500], 'm-', linewidth=0.8)
ax2.axhline(y=0.5, color='k', linestyle='--', alpha=0.5, label='理想值')
ax2.set_xlabel('符号索引')
ax2.set_ylabel('采样相位')
ax2.set_title('NCO采样相位收敛')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 子图3：同步后星座图
ax3 = axes[1, 0]
synced_loop_pts = synced_loop[::sps][:len(tx_symbols)]
ax3.plot(np.real(synced_loop_pts), np.imag(synced_loop_pts), 'bo', markersize=4, alpha=0.6)
ax3.plot([1, -1, -1, 1], [1, 1, -1, -1], 'r+', markersize=10)
ax3.set_xlabel('I（同相）')
ax3.set_ylabel('Q（正交）')
ax3.set_title('环路同步后星座图')
ax3.grid(True, alpha=0.3)
ax3.set_xlim(-1.5, 1.5)
ax3.set_ylim(-1.5, 1.5)

# 子图4：同步后眼图
ax4 = axes[1, 1]
for i in range(15):
    start = i * sps
    end = start + 2 * sps
    if end < len(synced_loop):
        ax4.plot(np.real(synced_loop[start:end]), 'b-', alpha=0.4)
ax4.set_xlabel('采样点')
ax4.set_ylabel('I分量幅度')
ax4.set_title('同步后眼图')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('timing_loop_demo.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n环路处理了 {len(synced_loop)} 个采样点")
print(f"鉴相器平均误差: {np.mean(np.abs(loop.error_history)):.4f}")

### 5.3 环路参数设计

定时同步环路的性能取决于以下参数：

| 参数 | 影响 | 设计考虑 |
|:---|:---|:---|
| **环路带宽** | 收敛速度 vs 噪声抑制 | 带宽大→收敛快但噪声大 |
| **阻尼系数** | 系统稳定性 | 通常设为 0.707 (临界阻尼) |
| **增益** | 误差放大倍数 | 影响跟踪精度 |
| **积分系数** | 稳态误差 | 影响同步精度 |

---

## 6. 综合演示：完整定时同步流程

### 6.1 系统框图

```
发射机 ──▶ AWGN信道 ──▶ 接收滤波器 ──▶ ADC ──▶ STO引入 ──▶ 定时同步 ──▶ 输出
```

### 6.2 演示代码

In [ ]:
def complete_timing_sync_demo(snr_db=20, sto_ppm=50, num_sym=200):
    """
    完整定时同步演示
    
    参数:
    -----------
    snr_db : float
        信噪比(dB)
    sto_ppm : float
        定时偏移（百万分之一符号周期）
    num_sym : int
        符号数量
    
    返回:
    -------
    results : dict
        包含中间结果和性能指标
    """
    sps = 16
    
    # 1. 生成QPSK信号
    signal_tx, symbols = generate_qpsk_signal(num_sym, sps, noise_var=0.0)
    
    # 2. 添加高斯白噪声
    snr_linear = 10 ** (snr_db / 10)
    signal_power = np.mean(np.abs(signal_tx) ** 2)
    noise_var = signal_power / snr_linear
    noise = np.sqrt(noise_var / 2) * (np.random.randn(len(signal_tx)) +
                                        1j * np.random.randn(len(signal_tx)))
    signal_rx = signal_tx + noise
    
    # 3. 引入STO（定时偏移）
    sto_samples = int(sto_ppm * 1e-6 * num_sym * sps)
    signal_sto = np.roll(signal_rx, sto_samples)
    
    # 4. 创建定时同步环路
    sync_loop = TimingRecoveryLoop(sps=sps, loop_bw=0.015)
    
    # 5. 处理信号
    synced_signal = sync_loop.process(signal_sto)
    
    # 6. 计算性能指标
    # 符号错误率（理想情况）
    synced_symbols = synced_signal[::sps][:num_sym]
    
    return {
        'synced_signal': synced_signal,
        'symbols': symbols,
        'sto_samples': sto_samples,
        'snr_db': snr_db,
        'error_history': sync_loop.error_history,
        'phase_history': sync_loop.phase_history
    }

# 运行完整演示
results = complete_timing_sync_demo(snr_db=15, sto_ppm=30, num_sym=300)

print("="*60)
print("完整定时同步流程演示结果")
print("="*60)
print(f"定时偏移: {results['sto_samples']} 采样点")
print(f"信噪比: {results['snr_db']} dB")
print(f"同步后采样数: {len(results['synced_signal'])}")

In [ ]:
# 绘图展示完整结果
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# 子图1：原始信号功率谱
ax1 = axes[0, 0]
f, psd = plt.psd(qpsk_signal, Nfft=1024, Fs=sps, noverlap=512)
ax1.plot(f, 10*np.log10(psd), 'b-', linewidth=1)
ax1.set_xlabel('归一化频率')
ax1.set_ylabel('功率谱密度(dB)')
ax1.set_title('QPSK信号功率谱')
ax1.grid(True, alpha=0.3)

# 子图2：定时误差收敛
ax2 = axes[0, 1]
ax2.plot(results['error_history'][:300], 'g-', linewidth=0.8)
ax2.axhline(y=0, color='k', linestyle='--', alpha=0.5)
ax2.set_xlabel('符号索引')
ax2.set_ylabel('误差')
ax2.set_title('鉴相器误差收敛')
ax2.grid(True, alpha=0.3)

# 子图3：采样相位收敛
ax3 = axes[0, 2]
ax3.plot(results['phase_history'][:300], 'm-', linewidth=0.8)
ax3.axhline(y=0.5, color='k', linestyle='--', alpha=0.5)
ax3.set_xlabel('符号索引')
ax3.set_ylabel('采样相位')
ax3.set_title('采样相位收敛')
ax3.grid(True, alpha=0.3)

# 子图4：原始信号星座图（STO后）
ax4 = axes[1, 0]
offset_idx = results['sto_samples']
ax4.plot(np.real(qpsk_signal[offset_idx:offset_idx+200:16]),
         np.imag(qpsk_signal[offset_idx:offset_idx+200:16]),
         'ro', markersize=5, alpha=0.5)
ax4.set_xlabel('I')
ax4.set_ylabel('Q')
ax4.set_title('STO后星座图（恶化）')
ax4.grid(True, alpha=0.3)
ax4.set_xlim(-1.5, 1.5)
ax4.set_ylim(-1.5, 1.5)

# 子图5：同步后星座图
ax5 = axes[1, 1]
synced_pts = results['synced_signal'][::sps][:len(results['symbols'])]
ax5.plot(np.real(synced_pts), np.imag(synced_pts), 'bo', markersize=4, alpha=0.6)
ax5.plot([1, -1, -1, 1], [1, 1, -1, -1], 'r+', markersize=10)
ax5.set_xlabel('I')
ax5.set_ylabel('Q')
ax5.set_title('同步后星座图（恢复）')
ax5.grid(True, alpha=0.3)
ax5.set_xlim(-1.5, 1.5)
ax5.set_ylim(-1.5, 1.5)

# 子图6：同步后眼图
ax6 = axes[1, 2]
for i in range(10):
    start = i * sps
    end = start + 2 * sps
    if end < len(results['synced_signal']):
        ax6.plot(np.real(results['synced_signal'][start:end]), 'b-', alpha=0.4)
ax6.set_xlabel('采样点')
ax6.set_ylabel('I分量')
ax6.set_title('同步后眼图（清晰）')
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('complete_sync_demo.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n完整定时同步演示完成，图像已保存")

---

## 总结与思考题

### 本章要点总结

1. **符号定时偏移（STO）** 是影响通信系统性能的关键因素之一，会导致符号间干扰和信噪比恶化。

2. **Early-Late 门算法** 通过比较符号前后两个采样点的能量差异来检测定时误差，实现简单但对噪声敏感。

3. **Gardner算法** 利用眼图特性进行定时误差检测，在中等信噪比下性能优异。

4. **插值滤波器** 是实现分数间隔采样的关键组件，线性插值适合低成本实现，立方插值适合高精度需求。

5. **定时同步环路** 通过闭环反馈机制实现采样相位的动态跟踪与调整。

### 深度思考题

**思考题 1**：在高速光通信系统中，由于采样率极高，Early-Late门算法的噪声敏感性会更加突出。请分析如何在保持低复杂度的前提下改进Early-Late算法以提高其噪声性能。

**提示**：可以考虑平均多个符号的误差结果，或使用加权方式减小噪声影响。

---

**思考题 2**：Gardner算法假设符号间存在理想的零点交叉特性，但在实际信道（如光纤）中可能存在模式耦合导致的非对称眼图。请讨论这种情况对Gardner算法性能的影响，以及可能的改进方向。

**提示**：考虑误差函数的不对称性，可能需要引入非对称的误差检测机制。

---

**思考题 3**：在多模光纤通信中，由于模间延迟差（差分群延迟，DGD），不同模式的光信号到达接收机的时间不同。请分析这与符号定时偏移的关系，以及定时同步算法在此场景下的特殊挑战。

**提示**：考虑空间复用系统中每个模式可能需要独立的定时同步环路。

---

## 参考资料

1. Gardner, F.M. (1986). "A BPSK/QPSK Timing-Error Detector for Sampled Receivers." IEEE Transactions on Communications, COM-34(5), 423-429.

2. Mengali, U., & D'Andrea, A.N. (1997). Synchronization Techniques for Digital Receivers. Plenum Press.

3. Proakis, J.G., & Salehi, M. (2007). Digital Communications. McGraw-Hill.

4. 楼的一种常见实现方式是在DSP或FPGA上采用并行处理结构，以适应高速数据处理的需求。